## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:0'

%load_ext autoreload
%autoreload 2

## Dataset

In [2]:
from datasets import NonlinearGaussian, MoG

n, d = 10000, 64                                 # < higher d, higher MI
true_rho = 0.7                                   # < higher rho, higher MI
case = 'MoG'                                      # < choose between ['1a', '1b', '2', '3a', '3b', '3c', 'MoG']

if case != 'MoG':
    dataset = NonlinearGaussian.NonlinearGaussian(n_samples=n, n_dims=d, rho=true_rho, mu=0, case=case)
    X0, Y0 = dataset.sample_data(n_samples = n)
    X, Y = dataset.transformation(X0, Y0)
    MI = dataset.true_mutual_info()              # we know GT MI
else:
    dataset = MoG.MoG(n_samples=n, n_dims=d, K=5, shifts=[-0.2, -0.1, 0, 0.3, 0.4], rhos=[-0.3, 0.5, 0.2, 0.4, 0.9])
    X, Y = dataset.sample_data(n_samples = n)
    MI = dataset.empirical_mutual_info()         # MI by MC estimate

X, Y = X.to(device), Y.to(device)
Z = torch.cat([X, Y], dim=1)
T = torch.ones(n, 2).to(device)

print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 6.893598159898664


## Hyperaparams

In [3]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## MI estimate

In [4]:
## Vector copula estimation
from estimators.VCE import VCE

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', dataset.empirical_mutual_info())
print('est MI:', estimator.MI(X, Y))


K components= 5 copula transform= True
nde type: FM
finished: t= 0 loss= 2.053354024887085 loss val= 2.0493078231811523 best val loss= 2.0493078231811523 best t= 0
finished: t= 126 loss= 1.6368829011917114 loss val= 1.633592963218689 best val loss= 1.6264445781707764 best t= 124
finished: t= 252 loss= 1.6041890382766724 loss val= 1.6642736196517944 best val loss= 1.6264445781707764 best t= 124


finished: t= 0 loss= 2.1056060791015625 loss val= 2.0539002418518066 best val loss= 2.0539002418518066 best t= 0
finished: t= 126 loss= 1.6193581819534302 loss val= 1.6457290649414062 best val loss= 1.6330037117004395 best t= 115
finished: t= 252 loss= 1.6223106384277344 loss val= 1.6812288761138916 best val loss= 1.6322401762008667 best t= 147


finished: t= 0 loss= 445.9299011230469 loss val= 443.349853515625 best val loss= 443.349853515625 best t= 0
finished: t= 101 loss= 84.70175170898438 loss val= 85.77184295654297 best val loss= 85.76822662353516 best t= 96
finished: t= 202 loss= 84.87009

In [9]:
n = len(X)

n_array = [50, 100, 500, 1000, 2500, 5000]
var_array = []
for n_data in n_array:
    var = []
    for i in range(8):
        idx = torch.randperm(n)
        XX, YY = X[idx][0:n_data], Y[idx][0:n_data]
        mi = estimator.MI(XX, YY)
        var.append(mi)
    var = np.array(var)
    var_array.append(var.std())
    print('n:', n_data, 'std', var.std())

n: 50 std 1.4173269904467731
n: 100 std 0.3176182909285189
n: 500 std 0.3114055410831705
n: 1000 std 0.2740780796129094
n: 2500 std 0.14512431389695613
n: 5000 std 0.12175468640130559
